# Combined recognition and robot control demo

This notebook combines several UGOT robot capabilities into one workflow:

1. **Initialize the robot and camera**
2. **Load built-in recognition models**
3. **Define helper functions** for line following, color recognition, and text recognition
4. **Test pose control** using a separate YOLO pose-control module
5. **Run a full example** that reacts to pose input, color signals, road markings, and text signs

The goal is to show how vision and motion can be chained together into one autonomous behavior.

## 1. Set up the robot and load recognition models

This cell imports the required libraries, connects to the robot, loads the UGOT vision models, and opens the camera.

A few important details:

- `ugot.UGOT()` creates the main robot control object.
- `initialize(...)` connects to the robot over the network.
- `load_models(...)` prepares the built-in recognition features.
- `open_camera()` starts the live camera feed that later functions will read.

In [ ]:
import time
import importlib
import cv2
import numpy as np
from ugot import ugot
import pose_yolo

# Reload the pose control module
importlib.reload(pose_yolo)
from pose_yolo import run_pose_control_inline

# Utilities for showing live camera frames directly inside the notebook
from IPython.display import display, clear_output, Image

# Create the robot controller object
got = ugot.UGOT()

# Connect to the UGOT robot using its IP address
got.initialize('192.168.1.189')

# Load the built-in computer vision models that will be used later in the notebook
got.load_models([
    'word_recognition',
    'color_recognition',
    'gesture',
    'line_recognition',
    'face_recognition',
    'face_attribute'
])

# Select the default line-tracking mode to single track
got.set_track_recognition_line(0)

# Open the camera stream so later functions can read live frames
got.open_camera()

print("Done!")

## 2. Built-in UGOT recognition features used here

In this notebook, the robot uses these built-in models:

- **Line recognition**: helps the robot follow road markings or tracks
- **Color recognition**: lets the robot react to colored objects or signals
- **Word recognition**: detects text such as signs

## 3. Helper functions for recognition and movement

This section creates three reusable functions:

- `line_follow(...)` keeps the robot centered on a detected line
- `color_rec()` reads the current detected color and shows an annotated camera frame
- `text_rec()` reads detected text and shows an annotated camera frame

These helper functions keep the main mission code shorter and easier to understand.

In [ ]:
def line_follow(mult=0.25, speed=35):
    """Follow the detected line by turning proportionally to the line offset."""
    # Read line-tracking information from the robot.
    # `offset` tells how far the detected line is from the center.
    # `type` describes the line/intersection pattern detected.
    # `x` and `y` are the detected line position coordinates.
    offset, type, x, y = got.get_single_track_total_info()

    # Convert the line offset into a turning speed.
    # Larger offset -> stronger rotation to re-center on the line.
    rotation_speed = int(offset * mult)

    # Move forward while rotating to stay aligned with the line.
    got.mecanum_move_xyz(x_speed=0, y_speed=speed, z_speed=rotation_speed)

    # Return the detection info so the main program can make decisions.
    return type, x, y


def color_rec():
    """Detect a color, draw the result on the current frame, and display it."""
    # Get the latest color recognition result from the UGOT model.
    color_info = got.get_color_total_info()

    # Read the current camera frame as encoded bytes.
    frame = got.read_camera_data()
    if frame is not None:
        # Convert raw bytes into an OpenCV image.
        nparr = np.frombuffer(frame, np.uint8)
        data = cv2.imdecode(nparr, cv2.IMREAD_COLOR)

        # If a color was detected, draw its name and bounding box.
        if color_info[0]:
            cv2.putText(
                data,
                color_info[0],
                (50, 40),
                cv2.FONT_HERSHEY_SIMPLEX,
                1,
                (255, 255, 255),
                2
            )

            c_x = color_info[2]
            c_y = color_info[3]
            h = color_info[4]
            w = color_info[5]

            cv2.rectangle(
                data,
                (int(c_x - w / 2), int(c_y - h / 2)),
                (int(c_x + w / 2), int(c_y + h / 2)),
                (0, 0, 255),
                2
            )

        # Show the annotated frame inside the notebook.
        _, jpeg = cv2.imencode('.jpg', data)
        clear_output(wait=True)
        display(Image(data=jpeg.tobytes()))

        # Return the detected color name (or None/empty if nothing was found).
        return color_info[0]



def text_rec():
    """Recognize text in the current frame, annotate it, and display it."""
    # Ask the UGOT word-recognition model for the current text result.
    text = got.get_words_result()

    # Read the current camera frame.
    frame = got.read_camera_data()
    if frame is not None:
        # Convert raw camera bytes into an OpenCV image.
        nparr = np.frombuffer(frame, np.uint8)
        data = cv2.imdecode(nparr, cv2.IMREAD_COLOR)

        # If text was found, draw it on top of the frame.
        if text:
            cv2.putText(
                data,
                text,
                (50, 40),
                cv2.FONT_HERSHEY_SIMPLEX,
                1,
                (255, 255, 255),
                2
            )

        # Display the annotated frame in the notebook.
        _, jpeg = cv2.imencode('.jpg', data)
        clear_output(wait=True)
        display(Image(data=jpeg.tobytes()))

        # Return the recognized text string.
        return text

## 4. Pose control module

Pose control is implemented in a separate Python file called `pose_yolo.py`.

The function `run_pose_control_inline(...)` uses a YOLO pose-estimation model to interpret body movement from a camera feed. Depending on the parameters, it can either:

- **Preview detections only** (`enable_robot=False`), or
- **Actively control the robot** (`enable_robot=True`)

This is useful for testing the pose-recognition system before combining it with the rest of the robot behaviors.

### Pose control test run

Use this cell to test whether pose detection is working correctly.

Because `enable_robot=False`, this is a safer debugging step: the model runs, but the robot does not physically respond yet. Change it to `True` to do it with the robot.

In [ ]:
try:
    # Run pose detection in test mode.
    # The model processes the webcam feed, but the robot itself is not moved.
    run_pose_control_inline(
        robot_ip='192.168.1.217',
        forward_speed=30,
        backward_speed=30,
        turn_speed=45,
        camera_index=0,
        model_path="yolov8n-pose.pt",
        up_margin_factor=0.6,
        down_margin_factor=0.6,
        min_conf=0.3,
        enable_robot=False,
        debounce_frames=2,
        max_frames=None
    )

except KeyboardInterrupt:
    print("Done")

## 5. Full mission example

This final section combines several behaviors into one sequence:

1. **Initialize the robot display and arm position**
2. **Run pose control** so the robot can be guided by body movement
3. **Wait for color signals** like a traffic light
4. **Follow line markings** and count intersections/roads
5. **Read a text sign** to decide where to turn
6. **Adjust direction** depending on how many roads were passed
7. **Stop and show a final screen state**

In [ ]:
try:
    # Set the screen to default background and move the mechanical arm
    # into starting position.
    got.screen_display_background(0)
    got.mechanical_joint_control(0, 45, 45, 500)

    # Stage 1: pose control 
    # Let the external pose-recognition module control the robot.
    run_pose_control_inline(
        forward_speed=30,
        backward_speed=30,
        turn_speed=45,
        camera_index=0,
        model_path="yolov8n-pose.pt",
        up_margin_factor=0.7,
        down_margin_factor=0.7,
        min_conf=0.3,
        enable_robot=True,
        debounce_frames=2,
        max_frames=None,
        got=got
    )

    # Stage 2: stop-light style color check 
    # Continuously detect colors and update the screen background based on
    # the detected signal. The loop exits when green is detected.
    while True:
        color = color_rec()

        if color == "Purple":
            got.screen_display_background(2)
        elif color == "Blue":
            got.screen_display_background(8)
        elif color == "Green":
            got.screen_display_background(6)
            time.sleep(1)
            break
        else:
            got.screen_display_background(0)

    # Stage 3: line following and sign detection
    # Count how many road/intersection markers are passed before the target
    # text sign is found.
    roads = 0

    while True:
        line_type, x, y = line_follow(mult=0.25, speed=35)

        # `line_type == 3` appears to indicate an intersection or special road marker.
        # `y > 380` ensures the marker is close enough before reacting.
        if line_type == 3 and y > 380:
            roads += 1

            # Move slightly forward, then turn to face the sign.
            got.mecanum_move_speed_times(0, 20, 25, 1)
            got.mecanum_turn_speed_times(2, 70, 90, 2)

            # Read text from the sign.
            text = text_rec()

            if text == "IMDA":
                # Target sign found: show a new background, turn around,
                # and exit this road-search loop.
                got.screen_display_background(1)
                got.mecanum_turn_speed_times(3, 70, 180, 2)
                break
            else:
                # Wrong sign: turn back and continue searching.
                got.mecanum_turn_speed_times(3, 70, 90, 2)

    # Stage 4: continue along the line after passing the sign
    while True:
        line_type, x, y = line_follow(mult=0.25, speed=30)

        if line_type == 3 and y > 390:
            # Move a little farther forward to fully clear the intersection,
            # then stop.
            for _ in range(9):
                line_follow(mult=0.25, speed=25)
            got.mecanum_stop()
            break

    # Stage 5: choose final turn based on how many roads were counted 
    if roads == 1:
        got.mecanum_turn_speed_times(3, 60, 90, 2)
    elif roads == 3:
        got.mecanum_turn_speed_times(2, 60, 90, 2)

    # Follow the line a little farther to finish the route.
    for _ in range(7):
        line_follow(mult=0.25, speed=20)

    got.mecanum_stop()

    # Stage 6: final state 
    # Show a finishing background and move forward slightly.
    got.screen_display_background(4)
    got.mecanum_move_speed_times(0, 30, 50, 1)

except KeyboardInterrupt:
    # Safety stop if the notebook is interrupted manually.
    got.mecanum_stop()
    print("Done")